# AgriTinyGPT Rung 4 — ~50M Parameters, Hydroponics Complete (from scratch)

This rung is hydroponics-only (no other domains mixed in), completing full A-Z
coverage: all 6 system types (wick, DWC, NFT, ebb & flow, drip, Kratky), all growing
media, complete macro/micronutrient deficiency list, ~19 crop types across leafy
greens, fruiting vegetables, herbs, vine crops, root vegetables, and flowers
(marigolds, petunias, orchids), plus light/CO2/water treatment/pests. All content is
original, written fresh rather than copied from any source. Architecture scaled to
roughly 50M parameters. Starting with this rung, each future rung focuses on one
domain at a time rather than mixing all six.

**Important: this rung needs a GPU.** Runtime -> Change runtime type -> T4 GPU -> Save.

Run all cells top to bottom.

## Cell 1 — Setup

In [ ]:
import torch, torch.nn as nn, re, base64
from torch.nn import functional as F

torch.manual_seed(1337)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


## Cell 2 — Build the dataset

Original agriculture corpus (paragraphs + Q&A), generated fresh — not copied from any site or PDF. Covers hydroponics, aeroponics, aquaponics, aquaculture, dairy farming, and microgreens.

In [ ]:
CORPUS_B64 = """..."""

text = base64.b64decode(CORPUS_B64).decode("utf-8")
with open("agri_corpus.txt", "w", encoding="utf-8") as f:
    f.write(text)

print("Corpus loaded. Characters:", len(text))
print(text[:500])


## Cell 3 — Tokenizer

Word-level tokenizer (character-level from the reference scripts doesn't scale to real vocabulary — this keeps things simple with no extra dependencies).

In [ ]:
def tokenize(s):
    return re.findall(r"\d+\.\d+|\w+|[^\w\s]", s.lower())

tokens = tokenize(text)
vocab = sorted(set(tokens))
vocab_size = len(vocab)
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}

def encode(s):
    return [stoi.get(w, 0) for w in tokenize(s)]

def decode(ids):
    words = [itos[i] for i in ids]
    out = ""
    for w in words:
        if re.match(r"^[^\w\s]$", w) and out:
            out += w
        else:
            out += (" " if out else "") + w
    return out

data = torch.tensor([stoi[w] for w in tokens], dtype=torch.long)
print("vocab_size:", vocab_size, "| total tokens:", len(data))


## Cell 4 — Model definition

Same architecture family as before, scaled up (larger embedding dim, more layers) to land close to 5,000,000 total parameters — Rung 2 of the progressive scale-up.

In [ ]:
block_size = 64
n_embd     = 840
n_head     = 8
n_layer    = 6

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = nn.MultiheadAttention(n_embd, n_head, batch_first=True, bias=False)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4*n_embd, bias=False), nn.GELU(),
                                  nn.Linear(4*n_embd, n_embd, bias=False))
    def forward(self, x):
        T = x.size(1)
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        a = self.ln1(x)
        x = x + self.attn(a, a, a, attn_mask=mask, need_weights=False)[0]
        return x + self.mlp(self.ln2(x))

class TinyAgriGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, n_embd)
        self.pos = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList([Block() for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.head.weight = self.tok.weight  # weight tying, no extra params
    def forward(self, idx, targets=None):
        pos = torch.arange(idx.size(1), device=idx.device)
        x = self.tok(idx) + self.pos(pos)
        for b in self.blocks: x = b(x)
        logits = self.head(self.ln_f(x))
        loss = None if targets is None else F.cross_entropy(
            logits.view(-1, vocab_size), targets.view(-1))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, n, temperature=0.8):
        for _ in range(n):
            logits, _ = self(idx[:, -block_size:])
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

model = TinyAgriGPT().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"vocab={vocab_size}  parameters={n_params:,}")


## Cell 5 — Training loop

Uses a lower learning rate with cosine decay and gradient clipping, since bigger
models need gentler, decaying updates to stay stable (a fixed high learning rate
caused this model's loss to improve early then drift back up and destabilize in
testing). The best checkpoint seen during training is kept, not just the final step,
as a safety net against any late-training drift.

In [ ]:
lr, steps, batch_size = 1e-3, 5000, 32

def get_batch():
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

opt = torch.optim.AdamW(model.parameters(), lr=lr)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps, eta_min=lr*0.05)
best_loss = float("inf")
best_state = None
for step in range(steps):
    x, y = get_batch()
    _, loss = model(x, y)
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    sched.step()
    if loss.item() < best_loss:
        best_loss = loss.item()
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    if step % 300 == 0:
        print(f"step {step:5d}  loss {loss.item():.3f}  lr {sched.get_last_lr()[0]:.2e}")

print("final loss (last step):", loss.item())
print("best loss seen during training:", best_loss)
model.load_state_dict(best_state)  # use the best checkpoint, not just the last step


## Cell 6 — Save checkpoint

Saves model weights, tokenizer, and config together (same pattern as `tiny_llm_1m.pt`). Uncomment the Drive line if you'd rather save there instead of downloading directly.

In [ ]:
torch.save({"model": model.state_dict(), "stoi": stoi, "itos": itos,
            "config": dict(block_size=block_size, n_embd=n_embd, n_head=n_head,
                            n_layer=n_layer, vocab_size=vocab_size)},
           "agri_tiny_llm_50m_hydro.pt")
print("saved agri_tiny_llm_50m_hydro.pt")

# --- Option A: download directly to your computer ---
from google.colab import files
files.download("agri_tiny_llm_50m_hydro.pt")

# --- Option B: save to Google Drive instead (uncomment to use) ---
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil
# shutil.copy("agri_tiny_llm_50m_hydro.pt", "/content/drive/MyDrive/agri_tiny_llm_50m_hydro.pt")


## Cell 7 — Evaluation

Reloads the checkpoint fresh (independent verification, same style as `test_tiny_llm_1m.py`) and generates text from agriculture seed prompts.

In [ ]:
ckpt = torch.load("agri_tiny_llm_50m_hydro.pt", map_location=device)
eval_model = TinyAgriGPT().to(device)
eval_model.load_state_dict(ckpt["model"])
eval_model.eval()

print("Reloaded parameter count:", sum(p.numel() for p in eval_model.parameters()))

seeds = [
    "Q: What is the ideal TDS for hydroponic lettuce?",
    "Q: What causes tipburn in hydroponic lettuce?",
    "Q: What EC should I use for hydroponic bell peppers during fruiting?",
    "Q: Why does my hydroponic spinach stop producing leaves and flower instead?",
    "Q: Can orchids be grown in a standing hydroponic nutrient solution?",
    "Q: Which hydroponic system is best for tomatoes and peppers?",
    "Q: What growing medium depth do hydroponic carrots need?",
    "Q: Why does mint need to be grown separately from other hydroponic crops?",
    "Q: What conditions do hydroponic peas need?",
]
for s in seeds:
    idx = torch.tensor([encode(s)], dtype=torch.long, device=device)
    out = eval_model.generate(idx, 40)
    print(f"\nSEED: {s}\n-> {decode(out[0].tolist())}")
